# 02 — SmarTRIZ DPO Training

Builds preference pairs from `rejected_dataset.jsonl` + `training_dataset_clean.json`, then fine-tunes the SFT model with DPO.

**Prerequisites:** Notebook 01 must be complete and `checkpoints/sft-7b/merged/` must exist.

In [ ]:
# ─── CONFIG — edit this cell only ────────────────────────
import os

MODEL_SIZE = '7b'
DRIVE_PATH = '/content/drive/MyDrive/smartriz/'

# Load API key from Colab secrets (preferred) or environment variable
try:
    from google.colab import userdata
    DEEPINFRA_API_KEY = userdata.get('DEEPINFRA_API_KEY') or os.getenv('DEEPINFRA_API_KEY', '')
except Exception:
    DEEPINFRA_API_KEY = os.getenv('DEEPINFRA_API_KEY', '')

if not DEEPINFRA_API_KEY:
    print('WARNING: DEEPINFRA_API_KEY not set — teacher generation will be skipped if needed.')
else:
    print('DEEPINFRA_API_KEY loaded successfully')

SFT_MODEL_DIR = f'{DRIVE_PATH}checkpoints/sft-{MODEL_SIZE}/merged/'
OUTPUT_DIR    = f'{DRIVE_PATH}checkpoints/dpo-{MODEL_SIZE}/'
DPO_DATASET   = f'{DRIVE_PATH}data/dpo_dataset.json'
CLEAN_DATASET = f'{DRIVE_PATH}data/training_dataset_clean.json'
REJECTED_PATH = f'{DRIVE_PATH}data/rejected_dataset.jsonl'
JUDGED_PATH   = f'{DRIVE_PATH}data/judged.jsonl'
TEACHER_MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Llama-70B'

LORA_R = 16 if MODEL_SIZE == '7b' else 32
LORA_ALPHA = 32 if MODEL_SIZE == '7b' else 64
MAX_LENGTH        = 3072
MAX_PROMPT_LENGTH = 1024
BETA = 0.1

# ── 14B memory warning ────────────────────────────────────
if MODEL_SIZE == '14b':
    print(
        '\n' + '='*70 + '\n'
        'WARNING: 14B DPO training memory requirements:\n'
        '  - SFT model (4-bit):       ~7 GB\n'
        '  - Explicit ref_model:      ~7 GB\n'
        '  - paged_adamw_8bit:        ~14 GB\n'
        '  - Activations (seq=3072):  ~12-15 GB\n'
        '  TOTAL ESTIMATED:           ~40-43 GB\n\n'
        '  A100 40GB: LIKELY OOM — reduce MAX_LENGTH to 2048 or use A100 80GB.\n'
        '  Recommended for 14B DPO: Colab Pro+ with A100 80GB.\n'
        '  To reduce memory: set MAX_LENGTH=2048 and MAX_PROMPT_LENGTH=768\n'
        + '='*70
    )
# ──────────────────────────────────────────────────────────

In [ ]:
!pip install -q transformers==4.45.2 peft==0.13.2 trl==0.11.4 bitsandbytes==0.44.1 accelerate==0.34.2 datasets==3.0.1 openai tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Resume detection: skip DPO pair building if already done
import json
from pathlib import Path

if Path(DPO_DATASET).exists():
    with open(DPO_DATASET) as f:
        dpo_pairs = json.load(f)
    print(f'DPO dataset already built: {len(dpo_pairs)} pairs — skipping build step')
else:
    dpo_pairs = None
    print('DPO dataset not found — will build from rejected + chosen')

checkpoint_dirs = sorted(Path(OUTPUT_DIR).glob('checkpoint-*')) if Path(OUTPUT_DIR).exists() else []
RESUME_FROM = str(checkpoint_dirs[-1]) if checkpoint_dirs else None
print(f'DPO checkpoint: {RESUME_FROM or "none — training from scratch"}')

In [ ]:
if dpo_pairs is None:
    from tqdm.auto import tqdm

    def load_jsonl(path):
        records = []
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return records

    rejected_raw = load_jsonl(REJECTED_PATH)
    judged_all   = load_jsonl(JUDGED_PATH)

    # ── Build chosen indexes from judged PASS records ─────────────────────
    judged_pass = [
        j for j in judged_all
        if j.get('meta', {}).get('judge_scores', {}).get('verdict') == 'PASS'
    ]
    judged_pass_by_id = {j['id']: j for j in judged_pass}

    # parent_seed_id → list of PASS records (for sibling matching)
    judged_pass_by_parent = {}
    for j in judged_pass:
        pid = j.get('meta', {}).get('parent_seed_id')
        if pid:
            judged_pass_by_parent.setdefault(pid, []).append(j)

    def best_chosen_for_parent(pid):
        candidates = judged_pass_by_parent.get(pid, [])
        if not candidates:
            return None
        return max(
            candidates,
            key=lambda j: j.get('meta', {}).get('judge_scores', {}).get('average', 0.0)
        )

    def format_assistant_from_judged(j):
        cp = j.get('contradiction_pair', {})
        principles = ', '.join(j.get('inventive_principles', []))
        return (
            f'<think>\n{j.get("reasoning_chain", "")}\n</think>\n'
            f'Contradiction:\n\n'
            f'Improving: {cp.get("improving_parameter", "")}\n'
            f'Worsening: {cp.get("worsening_parameter", "")}\n\n'
            f'Inventive Principles: {principles}\n'
            f'Solution:\n{j.get("solution", "")}'
        )

    def format_assistant_from_case(case):
        cp = case.get('contradiction_pair', {})
        principles = ', '.join(case.get('inventive_principles', []))
        return (
            f'<think>\n{case.get("reasoning_chain", "")}\n</think>\n'
            f'Contradiction:\n\n'
            f'Improving: {cp.get("improving_parameter", "")}\n'
            f'Worsening: {cp.get("worsening_parameter", "")}\n\n'
            f'Inventive Principles: {principles}\n'
            f'Solution:\n{case.get("solution", "")}'
        )

    # ── Identify usable rejected records ──────────────────────────────────
    # Exclude copy_sweep (no content) and matrix stage (content identical to PASS records)
    EXCLUDE_STAGES = {'copy_sweep', 'matrix'}
    matrix_case_ids = {
        r['case']['id']
        for r in rejected_raw
        if r.get('stage') == 'matrix' and r.get('case', {}).get('id')
    }

    candidate_rejected = [
        r for r in rejected_raw
        if r.get('stage') not in EXCLUDE_STAGES
        and r.get('case', {}).get('problem')
        and r.get('case', {}).get('solution')
        and r.get('case', {}).get('reasoning_chain')
        and r.get('case', {}).get('id') not in matrix_case_ids
        and r.get('case', {}).get('id') not in judged_pass_by_id  # skip cases that passed judgment (same content → identical pair)
    ]
    print(f'Usable rejected candidates: {len(candidate_rejected)}')

    # ── Also use judged FAIL records as rejected ───────────────────────────
    judged_fail = [
        j for j in judged_all
        if j.get('meta', {}).get('judge_scores', {}).get('verdict') == 'FAIL'
        and j.get('problem') and j.get('solution') and j.get('reasoning_chain')
    ]
    print(f'Judged FAIL candidates:     {len(judged_fail)}')

    dpo_pairs = []
    unmatched = []

    # Pass 1: rejected_dataset candidates
    # Cascade: exact case.id match → same parent_seed_id → skip
    for rej in tqdm(candidate_rejected, desc='matching rejected→chosen'):
        case    = rej['case']
        case_id = case.get('id', '')
        pid     = case.get('meta', {}).get('parent_seed_id')

        chosen_j = judged_pass_by_id.get(case_id)
        if chosen_j is None and pid:
            chosen_j = best_chosen_for_parent(pid)

        if chosen_j is None:
            unmatched.append({'source': 'rejected_dataset', 'record': rej})
            continue

        dpo_pairs.append({
            'prompt':   case.get('problem', ''),
            'chosen':   format_assistant_from_judged(chosen_j),
            'rejected': format_assistant_from_case(case),
        })

    # Pass 2: judged FAIL as rejected, best PASS sibling as chosen
    seen_fail_ids = set()
    for jf in tqdm(judged_fail, desc='pairing judged FAIL'):
        jf_id = jf.get('id', '')
        if jf_id in seen_fail_ids:
            continue
        pid      = jf.get('meta', {}).get('parent_seed_id')
        chosen_j = best_chosen_for_parent(pid) if pid else None
        if chosen_j is None or chosen_j.get('id') == jf_id:
            unmatched.append({'source': 'judged_fail', 'record': jf})
            continue
        dpo_pairs.append({
            'prompt':   jf.get('problem', ''),
            'chosen':   format_assistant_from_judged(chosen_j),
            'rejected': format_assistant_from_judged(jf),
        })
        seen_fail_ids.add(jf_id)

    # Deduplicate by prompt to avoid repeated problems
    seen_prompts = set()
    deduped = []
    for p in dpo_pairs:
        key = p['prompt'][:120]
        if key not in seen_prompts:
            seen_prompts.add(key)
            deduped.append(p)
    dpo_pairs = deduped

    # Defense layer: remove any identical chosen/rejected pairs (should be 0 after smart filter above)
    before_filter = len(dpo_pairs)
    dpo_pairs = [p for p in dpo_pairs if p['chosen'] != p['rejected']]
    filtered_identical = before_filter - len(dpo_pairs)
    if filtered_identical:
        print(f'WARNING: filtered {filtered_identical} identical chosen/rejected pairs (smart filter may have missed some)')

    assert len(dpo_pairs) > 0, 'DPO matching produced 0 pairs — check judged.jsonl path'
    assert filtered_identical == 0, f'Smart filter missed {filtered_identical} identical pairs — investigate judged_pass_by_id coverage'

    print(f'Matched pairs:   {len(dpo_pairs)}')
    print(f'Unmatched cases: {len(unmatched)} → teacher generation needed if total < 1500')

In [ ]:
# ── Generate chosen responses for unmatched cases via teacher (runs only if needed)
if dpo_pairs is not None and len(dpo_pairs) < 1500 and DEEPINFRA_API_KEY:
    from openai import OpenAI
    from tqdm.auto import tqdm

    client = OpenAI(api_key=DEEPINFRA_API_KEY,
                    base_url='https://api.deepinfra.com/v1/openai')
    TEACHER_SYSTEM = (
        'You are SmarTRIZ. Solve the engineering problem using TRIZ. '
        'Format your response exactly as:\n'
        '<think>\n[step by step reasoning]\n</think>\n'
        'Contradiction:\n\nImproving: [parameter]\nWorsening: [parameter]\n\n'
        'Inventive Principles: [list]\nSolution:\n[detailed solution]'
    )

    need = 1500 - len(dpo_pairs)
    print(f'Generating up to {min(need, len(unmatched))} chosen responses via teacher...')

    for entry in tqdm(unmatched[:need], desc='teacher generation'):
        src = entry['source']
        rec = entry['record']
        # Unpack problem correctly based on source
        if src == 'rejected_dataset':
            problem          = rec['case'].get('problem', '')
            rejected_response = format_assistant_from_case(rec['case'])
        else:  # judged_fail
            problem          = rec.get('problem', '')
            rejected_response = format_assistant_from_judged(rec)

        if not problem:
            continue
        try:
            resp = client.chat.completions.create(
                model=TEACHER_MODEL,
                messages=[
                    {'role': 'system', 'content': TEACHER_SYSTEM},
                    {'role': 'user',   'content': problem},
                ],
                temperature=0.3, max_tokens=2048,
            )
            dpo_pairs.append({
                'prompt':   problem,
                'chosen':   resp.choices[0].message.content,
                'rejected': rejected_response,
            })
        except Exception as e:
            print(f'Teacher API error: {e}')

    print(f'Total DPO pairs after teacher augmentation: {len(dpo_pairs)}')

# Save DPO dataset
if dpo_pairs is not None:
    with open(DPO_DATASET, 'w') as f:
        json.dump(dpo_pairs, f)
    print(f'Saved {len(dpo_pairs)} preference pairs to {DPO_DATASET}')

In [ ]:
# ── Load SFT merged model + apply fresh LoRA for DPO
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_DIR, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_DIR,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ── Load explicit reference model (frozen SFT base in 4-bit) ─────────────
# ref_model=None with trl 0.11.4 + PEFT has known gradient-checkpointing bugs
# that can produce inconsistent reference logprobs. An explicit frozen copy
# is safer and costs ~7 GB extra VRAM (negligible on A100 40GB for 7B).
ref_model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_DIR,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

assert not any(p.requires_grad for p in ref_model.parameters()), 'ref_model should be frozen'
print('Reference model loaded and frozen')

In [ ]:
# ── Build HF dataset and run DPOTrainer
from datasets import Dataset as HFDataset
from trl import DPOTrainer, DPOConfig

SYSTEM_PROMPT = (
    'You are SmarTRIZ, an expert engineering innovation assistant. '
    'Solve technical problems using TRIZ methodology. Identify the '
    'technical contradiction, select inventive principles from the '
    'Altshuller matrix, reason step by step, and propose a solution.'
)

def format_dpo_prompt(pair):
    prompt = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': pair['prompt']}],
        tokenize=False, add_generation_prompt=True
    )
    return {'prompt': prompt, 'chosen': pair['chosen'], 'rejected': pair['rejected']}

dpo_hf = HFDataset.from_list([format_dpo_prompt(p) for p in dpo_pairs])
print(f'DPO dataset size: {len(dpo_hf)}')

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=BETA,
    loss_type='sigmoid',
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to='none',
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=dpo_hf,
    tokenizer=tokenizer,
)

dpo_trainer.train(resume_from_checkpoint=RESUME_FROM)
print('DPO training complete')

In [ ]:
# ── Merge DPO LoRA into base and save
from pathlib import Path

merged_dpo = OUTPUT_DIR + 'merged/'
if not Path(merged_dpo + 'config.json').exists():
    print('Merging DPO LoRA weights...')
    model = model.merge_and_unload()
    model.save_pretrained(merged_dpo)
    tokenizer.save_pretrained(merged_dpo)
    print(f'DPO merged model saved to {merged_dpo}')
else:
    print(f'DPO merged model already exists at {merged_dpo}')

print('\nReady for Notebook 03 (GGUF conversion + evaluation)')